In [ ]:
# Lecture 24 — Broadcast Joins: shuffle join vs broadcast join
# Spark 101 · Module 7 · Lecture 24
#
# Import this file into a Databricks workspace as a notebook.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.
# In VS Code / PyCharm this stays a plain Python file — same source, two homes.
#
# BIG-DATA DEMO: use a big fact table so the wall-clock difference is real.
# Warn about DBU cost on screen and keep auto-termination on.

In [ ]:
print("Spark version:", spark.version)
# Databricks gives you `spark` for free — no SparkSession.builder needed.

In [ ]:
# --- A BIG fact table and a SMALL lookup table -----------------------------
from pyspark.sql.functions import col, broadcast
import time

big = spark.range(0, 30_000_000).withColumn("country_id", (col("id") % 4))   # millions
small = spark.createDataFrame(                                                # a handful
    [(0, "ES"), (1, "US"), (2, "FI"), (3, "FR")],
    ["country_id", "country"],
)
print("big rows:", big.count(), "| small rows:", small.count())

In [ ]:
# --- Force a SHUFFLE join: disable auto-broadcast so we can see the cost ----
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)   # -1 = off

shuffle_join = big.join(small, on="country_id", how="inner")
t0 = time.time()
shuffle_join.count()
print(f"shuffle join: {time.time() - t0:.1f}s")
shuffle_join.explain()   # look for `SortMergeJoin` + Exchange on BOTH sides

In [ ]:
# --- BROADCAST join: copy the small side to every executor -----------------
# Explicit hint so it's unmistakable (auto-broadcast is still off above).
broadcast_join = big.join(broadcast(small), on="country_id", how="inner")
t0 = time.time()
broadcast_join.count()
print(f"broadcast join: {time.time() - t0:.1f}s")
broadcast_join.explain()   # look for `BroadcastHashJoin` — NO shuffle of `big`

In [ ]:
# --- Back to automatic: small tables broadcast on their own ----------------
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)  # 10 MB default

auto = big.join(small, on="country_id", how="inner")
auto.explain()
# Under the threshold, Spark chooses BroadcastHashJoin for you — no hint needed.

In [ ]:
# SQL broadcast hint — same optimization, SQL syntax.
big.createOrReplaceTempView("big"); small.createOrReplaceTempView("small")
spark.sql("""
    SELECT /*+ BROADCAST(small) */ big.id, small.country
    FROM big JOIN small USING (country_id)
""").explain()      # BroadcastHashJoin

In [ ]:
# Inspect the auto-broadcast threshold (bytes).
print("threshold:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

In [ ]:
# Same result either way — correctness is identical, only the plan differs.
print("rows:", big.join(broadcast(small), "country_id").count())

In [ ]:
# Recap:
#   - Default join shuffles BOTH sides (SortMergeJoin) — wasteful for big×small.
#   - broadcast(small) copies the small side to every executor; big never moves.
#   - Confirm in the plan: BroadcastHashJoin = win, SortMergeJoin = shuffle.
#   - Only broadcast tables that fit in executor memory.
# Next lecture: the 5 common beginner mistakes (and their fixes).